<a href="https://colab.research.google.com/github/mjpepito/mids-266-NLP-final/blob/main/266_Final_MP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install transformers datasets accelerate scikit-learn pandas

In [2]:
import os
import re
import zipfile
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict

import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)

set_seed(42)

In [3]:
# from google.colab import files
# uploaded = files.upload()


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
def load_train_from_zip(zip_path: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()
        assert len(names) >= 1, "Zip is empty"
        with z.open(names[0]) as f:
            return pd.read_csv(f)

# train_df = load_train_from_zip("train.csv.zip")
train_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/w266/w266_final/train.csv")
val_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/w266/w266_final/validation.csv")
test_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/w266/w266_final/test.csv")

print(train_df.shape, val_df.shape, test_df.shape)
train_df.head()

(11118, 3) (1000, 3) (1000, 3)


,dialog,act,emotion
0,"['Say , Jim , how about going for a few beers ...",[3 4 2 2 2 3 4 1 3 4],[0 0 0 0 0 0 4 4 4 4]
1,"['Can you do push-ups ? '\n "" Of course I can ...",[2 1 2 2 1 1],[0 0 6 0 0 0]
2,"['Can you study with the radio on ? '\n ' No ,...",[2 1 2 1 1],[0 0 0 0 0]
3,['Are you all right ? '\n ' I will be all righ...,[2 1 1 1],[0 0 0 0]
4,"['Hey John , nice skates . Are they new ? '\n ...",[2 1 2 1 1 2 1 3 4],[0 0 0 0 0 6 0 6 0]


In [7]:
display(test_df)

,dialog,act,emotion
0,"['Hey man , you wanna buy some weed ? ' ' Some...",[3 2 3 4 3 4 3 2 3 4 2 3],[0 6 0 0 0 0 0 0 0 0 3 0]
1,['The taxi drivers are on strike again . ' ' W...,[1 2 1 1],[0 0 0 0]
2,"[""We've managed to reduce our energy consumpti...",[1 2 1 2 1 2 1],[0 0 0 0 0 0 0]
3,"['Believe it or not , tea is the most popular ...",[1 1 1 1 2 2 2 2 1 1 1 3 4 3],[0 0 0 0 0 0 0 0 0 4 0 0 4 4]
4,['What are your personal weaknesses ? '\n ' I ...,[2 1 2 1 2 1 2 1],[0 0 0 0 0 0 0 4]
...,...,...,...
995,"['Frank ’ s getting married , do you believe t...",[2 2 1 2 1 2 1],[0 6 0 0 0 0 0]
996,"['OK . Come back into the classroom , class . ...",[1 2 1 1 1 2 1],[0 0 0 5 0 0 0]
997,"['Do you have any hobbies ? ' ' Yes , I like c...",[2 1 2 1 2 1 1],[0 4 4 0 6 0 0]
998,"[""Jenny , what's wrong with you ? Why do you k...",[2 1 1],[0 0 0]


In [8]:
def parse_dialog(dialog_str: str) -> List[str]:
    """
    Extract utterances from strings like:
    ['hello' 'how are you' "fine"]
    """
    s = str(dialog_str).strip()
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1]

    matches = re.finditer(r"'([^']*)'|\"([^\"]*)\"", s)
    utterances = []
    for m in matches:
        utterances.append(m.group(1) if m.group(1) is not None else m.group(2))

    # basic cleanup
    utterances = [u.replace("\\n", " ").strip() for u in utterances if u.strip()]
    return utterances


def parse_label_list(label_str: str) -> List[int]:
    s = str(label_str).strip()
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1].strip()
    if not s:
        return []
    return [int(x) for x in s.split()]


def parse_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    bad_rows = 0

    for idx, row in df.iterrows():
        dialog = parse_dialog(row["dialog"])
        acts = parse_label_list(row["act"])
        emotions = parse_label_list(row["emotion"])

        if not (len(dialog) == len(acts) == len(emotions)):
            bad_rows += 1
            continue

        rows.append({
            "dialog_list": dialog,
            "act_list": acts,
            "emotion_list": emotions
        })

    parsed = pd.DataFrame(rows)
    print(f"Kept {len(parsed)} rows, filtered {bad_rows} malformed rows")
    return parsed


train_parsed = parse_dataframe(train_df)
val_parsed = parse_dataframe(val_df)
test_parsed = parse_dataframe(test_df)

Kept 11082 rows, filtered 36 malformed rows
Kept 997 rows, filtered 3 malformed rows
Kept 1000 rows, filtered 0 malformed rows


In [9]:
def flatten_dialogs(df: pd.DataFrame, context_size: int = 0) -> pd.DataFrame:
    examples = []

    for _, row in df.iterrows():
        utterances = row["dialog_list"]
        acts = row["act_list"]
        emotions = row["emotion_list"]

        for i, utt in enumerate(utterances):
            if context_size > 0:
                start = max(0, i - context_size)
                context_utts = utterances[start:i]
                text = " [SEP] ".join(context_utts + [utt])
            else:
                text = utt

            examples.append({
                "text": text,
                "act_label": acts[i],
                "emotion_label": emotions[i]
            })

    return pd.DataFrame(examples)


# Start simple: no context
train_flat = flatten_dialogs(train_parsed, context_size=0)
val_flat = flatten_dialogs(val_parsed, context_size=0)
test_flat = flatten_dialogs(test_parsed, context_size=0)

print(train_flat.shape, val_flat.shape, test_flat.shape)
train_flat.head()

(86736, 3) (8019, 3) (7740, 3)


,text,act_label,emotion_label
0,"Say , Jim , how about going for a few beers af...",3,0
1,You know that is tempting but is really not go...,4,0
2,What do you mean ? It will help us to relax .,2,0
3,Do you really think so ? I don't . It will jus...,2,0
4,I guess you are right.But what shall we do ? I...,2,0


In [10]:
TASK = "act"   # change to "emotion" for second run

label_col = "act_label" if TASK == "act" else "emotion_label"

num_labels = int(max(
    train_flat[label_col].max(),
    val_flat[label_col].max(),
    test_flat[label_col].max()
)) + 1

print("Task:", TASK)
print("Num labels:", num_labels)
print(train_flat[label_col].value_counts().sort_index())

Task: act
Num labels: 5
act_label
1    39654
2    24846
3    14187
4     8049
Name: count, dtype: int64


In [11]:
def to_hf_dataset(df: pd.DataFrame, label_col: str) -> Dataset:
    data = df[["text", label_col]].copy()
    data = data.rename(columns={label_col: "labels"})
    return Dataset.from_pandas(data, preserve_index=False)

train_ds = to_hf_dataset(train_flat, label_col)
val_ds = to_hf_dataset(val_flat, label_col)
test_ds = to_hf_dataset(test_flat, label_col)

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/86736 [00:00<?, ? examples/s]

Map:   0%|          | 0/8019 [00:00<?, ? examples/s]

Map:   0%|          | 0/7740 [00:00<?, ? examples/s]

In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted")
    }

In [13]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

training_args = TrainingArguments(
    output_dir=f"./{TASK}_bert_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.451500,0.515443,0.804090,0.731537,0.790394
2,0.403957,0.500977,0.813443,0.762842,0.809023


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.451500,0.515443,0.804090,0.731537,0.790394
2,0.403957,0.500977,0.813443,0.762842,0.809023
3,0.318013,0.568974,0.812820,0.760927,0.807529


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=16263, training_loss=0.3991664893980624, metrics={'train_runtime': 1323.8345, 'train_samples_per_second': 196.556, 'train_steps_per_second': 12.285, 'total_flos': 5709943254495168.0, 'train_loss': 0.3991664893980624, 'epoch': 3.0})

In [15]:
val_metrics = trainer.evaluate(val_ds)
test_metrics = trainer.evaluate(test_ds)

print("Validation:", val_metrics)
print("Test:", test_metrics)

test_preds = trainer.predict(test_ds)
y_true = test_ds["labels"]
y_pred = np.argmax(test_preds.predictions, axis=-1)

print(classification_report(y_true, y_pred, digits=4))

Validation: {'eval_loss': 0.5013903975486755, 'eval_accuracy': 0.8131936650455169, 'eval_macro_f1': 0.7626922049594842, 'eval_weighted_f1': 0.8088070161696792, 'eval_runtime': 6.6643, 'eval_samples_per_second': 1203.281, 'eval_steps_per_second': 37.663, 'epoch': 3.0}
Test: {'eval_loss': 0.44201141595840454, 'eval_accuracy': 0.8408268733850129, 'eval_macro_f1': 0.7720350716387435, 'eval_weighted_f1': 0.8383755975517896, 'eval_runtime': 6.4348, 'eval_samples_per_second': 1202.826, 'eval_steps_per_second': 37.608, 'epoch': 3.0}
              precision    recall  f1-score   support

           1     0.8663    0.8749    0.8706      3534
           2     0.9309    0.9507    0.9407      2210
           3     0.7346    0.7535    0.7439      1278
           4     0.5837    0.4903    0.5329       718

    accuracy                         0.8408      7740
   macro avg     0.7789    0.7673    0.7720      7740
weighted avg     0.8368    0.8408    0.8384      7740



In [17]:
slang_df = pd.read_csv("hf://datasets/MLBtrio/genz-slang-dataset/all_slangs.csv")
slang_df.head()

,Slang,Description,Example,Context
0,W,Shorthand for win,"Got the job today, big W!",Typically used in conversations to celebrate s...
1,L,Shorthand for loss/losing,"I forgot my wallet at home, that’s an L.",Often used when referring to a failure or mish...
2,L+ratio,Response to a comment or action on the interne...,Your tweet got 5 likes and 100 replies calling...,Popularized on social media platforms to signi...
3,Dank,excellent or of very high quality,That meme is so dank!,Commonly used in internet slang to refer to me...
4,Cheugy,Derogatory term for Millennials. Used when mil...,"That phrase is so cheugy, no one says that any...",Used to refer to things that were once popular...


## Explore Gen Z Slang Dataset

### Subtask:
Inspect the `slang_df` to understand its structure. This involves displaying the first few rows with `slang_df.head()`, checking column information with `slang_df.info()`, and looking at descriptive statistics with `slang_df.describe(include='object')`.


**Reasoning**:
To understand the structure of `slang_df`, I will display a concise summary of its columns and data types using `info()` and then generate descriptive statistics for its object columns using `describe(include='object')`.



In [18]:
print('DataFrame Info:')
slang_df.info()

print('\nDescriptive Statistics for Object Columns:')
slang_df.describe(include='object')

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1779 entries, 0 to 1778
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Slang        1779 non-null   object
 1   Description  1779 non-null   object
 2   Example      1779 non-null   object
 3   Context      1779 non-null   object
dtypes: object(4)
memory usage: 55.7+ KB

Descriptive Statistics for Object Columns:


,Slang,Description,Example,Context
count,1779,1779,1779,1779
unique,1605,1648,1773,1773
top,LOL,I love you,"Goodnight, ILY!",Used to agree emphatically with someone’s stat...
freq,4,5,3,2


## Define Translation Strategy and Prepare Data

Based on the Gen Z slang dataset exploration, define the strategy for 'converting language into Gen Z'. This could involve direct word/phrase substitution, sentence restructuring, or generating Gen Z-style sentences. Then, create a dataset of (source sentence, target Gen Z sentence) pairs. This will be the core training data for the translator. This might involve creating synthetic data if the slang dataset doesn't directly provide pairs, and leveraging the DailyDialog dataset as a source of 'normal' language.


**Reasoning**:
The subtask requires defining a translation strategy and preparing a dataset of (source sentence, target Gen Z sentence) pairs from the `slang_df`. This involves iterating through the DataFrame, using the 'Example' column as the target Gen Z sentence, and creating a 'source_sentence' by replacing the slang term with its description. I will use regular expressions to ensure accurate and case-insensitive replacement of whole words to create the source sentence.



In [19]:
import re

gen_z_translation_data = []

for index, row in slang_df.iterrows():
    slang_term = row['Slang']
    description = row['Description']
    target_genz_sentence = row['Example']

    # Create a regex pattern to match the whole word, case-insensitive
    # Use re.escape to handle special characters in slang_term
    # Add word boundaries (\b) to ensure whole word replacement
    pattern = r'\b' + re.escape(slang_term) + r'\b'

    # Replace slang_term with its description to create the source sentence
    # Use re.IGNORECASE for case-insensitive matching
    source_sentence = re.sub(pattern, description, target_genz_sentence, flags=re.IGNORECASE)

    gen_z_translation_data.append({
        'source_sentence': source_sentence,
        'target_genz_sentence': target_genz_sentence
    })

gen_z_translation_df = pd.DataFrame(gen_z_translation_data)

print("Generated Gen Z Translation Data:")
gen_z_translation_df.head()

Generated Gen Z Translation Data:


,source_sentence,target_genz_sentence
0,"Got the job today, big Shorthand for win!","Got the job today, big W!"
1,"I forgot my wallet at home, that’s an Shorthan...","I forgot my wallet at home, that’s an L."
2,Your tweet got 5 likes and 100 replies calling...,Your tweet got 5 likes and 100 replies calling...
3,That meme is so excellent or of very high qual...,That meme is so dank!
4,That phrase is so Derogatory term for Millenni...,"That phrase is so cheugy, no one says that any..."


## ^ The above "Source Sentences" made with substitution (naive approach) are terrible, not natural at all. Pivoting to using normal source language and generating the Gen Z Slang

# Task
Implement a Gen Z language translator by preparing a custom translation dataset using the 'Slang' terms, 'Description', 'Context', and 'Example' from `slang_df` (from "hf://datasets/MLBtrio/genz-slang-dataset/all_slangs.csv") and natural language sentences from the DailyDialog dataset (from "/content/drive/MyDrive/Colab Notebooks/w266/w266_final/train.csv", "/content/drive/MyDrive/Colab Notebooks/w266/w266_final/validation.csv", and "/content/drive/MyDrive/Colab Notebooks/w266/w266_final/test.csv"), fine-tuning a sequence-to-sequence model, and evaluating its performance.

## Revise Data Preparation Strategy (Slang Dataset)

### Subtask:
Create a dataset of source-target sentence pairs where the source explains the slang using its description and context, and the target is an example sentence using the slang.


**Reasoning**:
I need to implement the revised data preparation strategy as outlined in the instructions. This involves iterating through the `slang_df`, constructing a new `source_sentence` by combining slang, description, and context, and setting the `target_genz_sentence` to the example. Finally, I will create a new DataFrame and display its head.



In [20]:
gen_z_translation_data_revised = []

for index, row in slang_df.iterrows():
    slang_term = row['Slang']
    description = row['Description']
    context = row['Context']
    target_genz_sentence = row['Example']

    # Construct the source sentence by combining slang, description, and context
    source_sentence = f"The slang term '{slang_term}' means: {description}. This is typically used in situations like: {context}"

    gen_z_translation_data_revised.append({
        'source_sentence': source_sentence,
        'target_genz_sentence': target_genz_sentence
    })

gen_z_translation_df_revised = pd.DataFrame(gen_z_translation_data_revised)

print("Generated Revised Gen Z Translation Data:")
gen_z_translation_df_revised.head()

Generated Revised Gen Z Translation Data:


,source_sentence,target_genz_sentence
0,The slang term 'W' means: Shorthand for win. T...,"Got the job today, big W!"
1,The slang term 'L' means: Shorthand for loss/l...,"I forgot my wallet at home, that’s an L."
2,The slang term 'L+ratio' means: Response to a ...,Your tweet got 5 likes and 100 replies calling...
3,The slang term 'Dank' means: excellent or of v...,That meme is so dank!
4,The slang term 'Cheugy' means: Derogatory term...,"That phrase is so cheugy, no one says that any..."


## Integrate DailyDialog for Natural Language Expansion

### Subtask:
Create a dataset of 'natural language' to 'Gen Z language' translation pairs by applying a selection of slang substitutions from `slang_df` to sentences from the DailyDialog dataset (`train_flat`).


**Reasoning**:
I need to initialize the `slang_substitutions` dictionary with relevant mappings from the `slang_df` as specified in the instructions to prepare for generating Gen Z translation pairs.



In [21]:
import re

# 1. Create a dictionary to map 'normal' words/phrases to their 'Gen Z' slang equivalents
slang_substitutions = {
    "lie": "cap",
    "no lie": "no cap",
    "win": "W",
    "loss": "L",
    "excellent": "dank",
    "okay": "bet",
    "disappearing without explanation": "ghosting",
    "over the top": "extra",
    "cool": "lit", # Added from slang_df example
    "confused": "muddle", # Added from slang_df example
    "very good": "bussin", # Added from slang_df example
    "charisma": "rizz", # Added from slang_df example (rizz is more charisma, but can be used as 'true')
    "sleeping": "ZZZZ", # Mapping 'sleeping' to 'ZZZZ'
    "crazy": "wild", # Example of a common Gen Z usage
    "friend": "bestie", # Example of a common Gen Z usage
    "amazing": "slay", # Example of a common Gen Z usage
    "annoying": "cringe", # Example of a common Gen Z usage
    "attractive": "simp", # Example of a common Gen Z usage
    "boring": "mid", # Example of a common Gen Z usage
    "funny": "hilarious", # Example of a common Gen Z usage
    "stressed": "panicked", # Example of a common Gen Z usage
    "excited": "hyped", # Example of a common Gen Z usage
    "tired": "zonked", # Example of a common Gen Z usage
    "goodbye": "peace out", # Example of a common Gen Z usage
    "hungry": "hangry", # Example of a common Gen Z usage
    "money": "racks", # Example of a common Gen Z usage
    "party": "kickback", # Example of a common Gen Z usage
    "problem": "dilemma", # Example of a common Gen Z usage
    "scared": "shook", # Example of a common Gen Z usage
    "secret": "tea", # Example of a common Gen Z usage
    "smart": "brainy", # Example of a common Gen Z usage
    "upset": "salty", # Example of a common Gen Z usage
    "weird": "sus", # Example of a common Gen Z usage
    "work": "grind", # Example of a common Gen Z usage
    "yes": "yass", # Example of a common Gen Z usage
    "you only live once": "yolo", # Example of a common Gen Z usage
    "i love you": "ily", # From slang_df example
    "goodnight": "gn", # From slang_df example
    "good game": "gg", # From slang_df example
    "be right back": "brb", # From slang_df example
    "laughing out loud": "lol", # From slang_df example
    "very good": "fire", # From slang_df
    "no problem": "np", # From slang_df
    "in my opinion": "imo", # From slang_df
    "for your information": "fyi", # From slang_df
    "talk to you later": "ttyl", # From slang_df
    "by the way": "btw", # From slang_df
    "rolling on the floor laughing": "rofl", # From slang_df
    "as soon as possible": "asap", # From slang_df
    "what the fuck": "wtf", # From slang_df
    "oh my god": "omg" # From slang_df
}

# 2. Initialize an empty list
daily_dialog_genz_data = []

# Get unique sentences from train_flat to avoid redundant processing
unique_original_sentences = train_flat['text'].unique()

# 3. Iterate through each unique sentence in the `text` column of the `train_flat` DataFrame
for original_sentence in unique_original_sentences:
    # 4a. Initialize a variable `gen_z_sentence` with the `original_sentence`.
    gen_z_sentence = original_sentence
    substituted = False # Flag to check if any substitution occurred

    # 4b. Iterate through the `slang_substitutions` dictionary
    for normal_word_phrase, slang_term in slang_substitutions.items():
        # 4c. Construct a regular expression pattern to match the `normal_word_phrase` as a whole word, case-insensitively
        pattern = r'\b' + re.escape(normal_word_phrase) + r'\b'

        # 4d. Replace all occurrences of the `normal_word_phrase` with the corresponding `slang_term`
        new_gen_z_sentence = re.sub(pattern, slang_term, gen_z_sentence, flags=re.IGNORECASE)

        if new_gen_z_sentence != gen_z_sentence:
            substituted = True
            gen_z_sentence = new_gen_z_sentence

    # 4e. If `gen_z_sentence` is different from `original_sentence`, add the pair
    if substituted:
        daily_dialog_genz_data.append({
            'source_sentence': original_sentence,
            'target_genz_sentence': gen_z_sentence
        })

# 5. Convert the `daily_dialog_genz_data` list into a Pandas DataFrame
daily_dialog_genz_df = pd.DataFrame(daily_dialog_genz_data)

# 6. Display the first few rows of `daily_dialog_genz_df`
print("Generated DailyDialog Gen Z Translation Data:")
daily_dialog_genz_df.head()

Generated DailyDialog Gen Z Translation Data:


,source_sentence,target_genz_sentence
0,"Sounds great to me ! If they are willing , we ...","Sounds great to me ! If they are willing , we ..."
1,Let ’ s see . . . you ’ re a Capricorn . It sa...,Let ’ s see . . . you ’ re a Capricorn . It sa...
2,That ’ s bogus . I don't feel any stress at wo...,That ’ s bogus . I don't feel any stress at gr...
3,"Yes , he is . He loves the girl very much .","yass , he is . He loves the girl very much ."
4,What's the problem ?,What's the dilemma ?


**Reasoning**:
I need to combine the two generated DataFrames, `gen_z_translation_df_revised` (from the slang dataset) and `daily_dialog_genz_df` (from the DailyDialog dataset with slang substitutions), into a single DataFrame to create a comprehensive dataset for Gen Z translation.



In [22]:
combined_genz_df = pd.concat([gen_z_translation_df_revised, daily_dialog_genz_df], ignore_index=True)

print("Combined Gen Z Translation Data:")
print(combined_genz_df.head())
print("\nShape of combined_genz_df:", combined_genz_df.shape)

Combined Gen Z Translation Data:
                                     source_sentence  \
0  The slang term 'W' means: Shorthand for win. T...   
1  The slang term 'L' means: Shorthand for loss/l...   
2  The slang term 'L+ratio' means: Response to a ...   
3  The slang term 'Dank' means: excellent or of v...   
4  The slang term 'Cheugy' means: Derogatory term...   

                                target_genz_sentence  
0                          Got the job today, big W!  
1           I forgot my wallet at home, that’s an L.  
2  Your tweet got 5 likes and 100 replies calling...  
3                              That meme is so dank!  
4  That phrase is so cheugy, no one says that any...  

Shape of combined_genz_df: (11576, 2)


**Reasoning**:
Now that the training data is combined, I need to split it into training and testing sets to evaluate the model's performance. I will use 10% of the combined data as a test set.



In [23]:
from sklearn.model_selection import train_test_split

# Split the combined dataset into training and test sets
train_df, test_df = train_test_split(combined_genz_df, test_size=0.1, random_state=42)

print(f"Shape of training data: {train_df.shape}")
print(f"Shape of test data: {test_df.shape}")

# Display first few rows of the training data
print("\nTraining Data Head:")
print(train_df.head())

# Display first few rows of the test data
print("\nTest Data Head:")
print(test_df.head())

Shape of training data: (10418, 2)
Shape of test data: (1158, 2)

Training Data Head:
                                         source_sentence  \
9792   Yes , true , but how about other personal qual...   
11377  Yes . Let ’ s say Tuesday at noon . Bring your...   
10241              Yes , I will be home this afternoon .   
1839   Yes , I agree with you . When I retired I will...   
4812   Yes ! Somebody up there loves me ! I knew it w...   

                                    target_genz_sentence  
9792   yass , true , but how about other personal qua...  
11377  yass . Let ’ s say Tuesday at noon . Bring you...  
10241             yass , I will be home this afternoon .  
1839   yass , I agree with you . When I retired I wil...  
4812   yass ! Somebody up there loves me ! I knew it ...  

Test Data Head:
                                        source_sentence  \
2611  Yes . Did you know that the senior class presi...   
7965                                    Yes , ma ’ am ?   
4419 

In [24]:
from datasets import Dataset
from transformers import AutoTokenizer

# Convert pandas DataFrames to Hugging Face Dataset objects
train_translation_ds = Dataset.from_pandas(train_df, preserve_index=False)
test_translation_ds = Dataset.from_pandas(test_df, preserve_index=False)

# Load the tokenizer for a sequence-to-sequence model (e.g., T5)
model_checkpoint = "t5-small"
translation_tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    # Tokenize source sentences
    model_inputs = translation_tokenizer(examples['source_sentence'], max_length=128, truncation=True)

    # Tokenize target sentences
    labels = translation_tokenizer(text_target=examples['target_genz_sentence'], max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply the preprocessing function to the datasets
tokenized_train_translation_ds = train_translation_ds.map(preprocess_function, batched=True)
tokenized_test_translation_ds = test_translation_ds.map(preprocess_function, batched=True)

print("Tokenized Training Dataset example:", tokenized_train_translation_ds[0])
print("Tokenized Test Dataset example:", tokenized_test_translation_ds[0])

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/10418 [00:00<?, ? examples/s]

Map:   0%|          | 0/1158 [00:00<?, ? examples/s]

Tokenized Training Dataset example: {'source_sentence': 'Yes , true , but how about other personal qualities ?', 'target_genz_sentence': 'yass , true , but how about other personal qualities ?', 'input_ids': [2163, 3, 6, 1176, 3, 6, 68, 149, 81, 119, 525, 10596, 3, 58, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [3, 63, 9, 7, 7, 3, 6, 1176, 3, 6, 68, 149, 81, 119, 525, 10596, 3, 58, 1]}
Tokenized Test Dataset example: {'source_sentence': "Yes . Did you know that the senior class president works for the class for life ? The job doesn't stop when we graduate .", 'target_genz_sentence': "yass . Did you know that the senior class president works for the class for life ? The job doesn't stop when we graduate .", 'input_ids': [2163, 3, 5, 3963, 25, 214, 24, 8, 2991, 853, 2753, 930, 21, 8, 853, 21, 280, 3, 58, 37, 613, 744, 31, 17, 1190, 116, 62, 5213, 3, 5, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

**Reasoning**:
The previous step's tokenization likely completed despite the truncated output. The next logical step is to prepare the sequence-to-sequence model for fine-tuning by loading the appropriate model, setting up the data collator, defining training arguments, and initializing the Trainer.



In [29]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

# Load the T5 model for sequence-to-sequence tasks
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# Define the data collator for sequence-to-sequence tasks
data_collator = DataCollatorForSeq2Seq(tokenizer=translation_tokenizer, model=model_checkpoint)

# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./genz_translator",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    logging_steps=100,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", # T5 typically uses loss for evaluation
    greater_is_better=False
)

# Initialize the Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_translation_ds,
    eval_dataset=tokenized_test_translation_ds, # Using test_ds for evaluation during training for now
    processing_class=translation_tokenizer,
    data_collator=data_collator,
)

print("Trainer initialized. Ready to train.")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Trainer initialized. Ready to train.


In [30]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.466903,0.402058
2,0.393709,0.324824
3,0.355444,0.310624


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=1956, training_loss=0.4892975343029436, metrics={'train_runtime': 269.502, 'train_samples_per_second': 115.969, 'train_steps_per_second': 7.258, 'total_flos': 520106912120832.0, 'train_loss': 0.4892975343029436, 'epoch': 3.0})

## Evaluate Model Performance

### Subtask:
Evaluate the fine-tuned model's performance on the test set using relevant metrics such as BLEU score and ROUGE. Additionally, conduct qualitative analysis by testing the model with various input sentences to assess the naturalness and accuracy of the Gen Z translations. This will involve importing and using the `evaluate` library.


**Reasoning**:
The first step for evaluating model performance with BLEU and ROUGE scores, as well as for qualitative analysis, is to install the `evaluate` library. This is a prerequisite for loading the necessary metrics and utilities.



In [38]:
!pip -q install evaluate
print("Installed 'evaluate' library.")

Installed 'evaluate' library.


In [35]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=20483bd3d5e3de40c8302c7a39cb7541e4d9b8420f52a523921e0d400dfec151
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [36]:
import evaluate

# 1. & 2. Load the 'bleu' and 'rouge' metrics
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

print("Loaded BLEU and ROUGE metrics.")

# 3. Generate predictions from the trainer on the test set
predictions = trainer.predict(tokenized_test_translation_ds)
print("Generated predictions.")

# Extract logits and labels
pred_logits = predictions.predictions
pred_labels = predictions.label_ids

# Replace -100 in labels as we did in the data collator
pred_labels = np.where(pred_labels != -100, pred_labels, translation_tokenizer.pad_token_id)

# 4. Decode the generated predictions and the true labels
decoded_preds = translation_tokenizer.batch_decode(pred_logits, skip_special_tokens=True)
decoded_labels = translation_tokenizer.batch_decode(pred_labels, skip_special_tokens=True)

print("Decoded predictions and true labels.")

# Prepare references for ROUGE (list of lists of strings)
# ROUGE expects references as a list of lists of strings, where each inner list contains one or more reference texts for a given hypothesis.
# Since we only have one reference per hypothesis, each inner list will have one element.
formatted_decoded_labels = [[label] for label in decoded_labels]

# 5. Compute the BLEU score
bleu_results = bleu_metric.compute(predictions=decoded_preds, references=formatted_decoded_labels)
print("\nBLEU Score:", bleu_results)

# 6. Compute the ROUGE score
rouge_results = rouge_metric.compute(predictions=decoded_preds, references=formatted_decoded_labels)
print("\nROUGE Score:", rouge_results)

# 7. For qualitative analysis, select a few sample source_sentences
sample_sentences = [
    "That is excellent exercise and fun.",
    "I will be home this afternoon.",
    "What's the problem?",
    "Are you all right?",
    "No problem, I can do it."
]

print("\nQualitative Analysis:")
for sentence in sample_sentences:
    # 8. Tokenize these sample sentences
    inputs = translation_tokenizer(sentence, return_tensors="pt", max_length=128, truncation=True)

    # Move inputs to the same device as the model (e.g., GPU if available)
    input_ids = inputs.input_ids.to(model.device)
    attention_mask = inputs.attention_mask.to(model.device)

    # 9. Use the model.generate() method to get the Gen Z translation
    generated_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=128,
        num_beams=4, # Use beam search for better quality
        early_stopping=True
    )

    # 10. Decode the generated output IDs back into text
    genz_translation = translation_tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    print(f"Original: {sentence}")
    print(f"Gen Z: {genz_translation}\n")


Loaded BLEU and ROUGE metrics.


Generated predictions.
Decoded predictions and true labels.

BLEU Score: {'bleu': 0.5377956240415265, 'precisions': [0.9035766255648311, 0.8705773594419699, 0.8588585792756727, 0.8478170478170478], 'brevity_penalty': 0.6181846602844522, 'length_ratio': 0.6752340073434349, 'translation_length': 13057, 'reference_length': 19337}

ROUGE Score: {'rouge1': np.float64(0.7439872625087149), 'rouge2': np.float64(0.6886326589086917), 'rougeL': np.float64(0.7416823652159881), 'rougeLsum': np.float64(0.7418867818106247)}

Qualitative Analysis:
Original: That is excellent exercise and fun.
Gen Z: That is excellent exercise and fun.

Original: I will be home this afternoon.
Gen Z: I will be home this afternoon.

Original: What's the problem?
Gen Z: What's the dilemma?

Original: Are you all right?
Gen Z: Are you all right?

Original: No problem, I can do it.
Gen Z: No dilemma, I can do it.

